In [1]:
import pinecone
import pandas as pd
#pinecone.init(api_key=open("Pineconekey.txt","r").read(), environment="gcp-starter")
#pc = pinecone(api_key=open("Pineconekey.txt","r").read(), environment="gcp-starter")   
import os
from pinecone import Pinecone, ServerlessSpec, PodSpec

import configparser; config = configparser.RawConfigParser()
config.read('keys//keys.ini')
api_key = config['keys']['Pinecone']

pc = Pinecone(api_key=api_key)


In [2]:
#pc.delete_index("alice")

In [3]:
# pc.create_index(
#   name="alice",
#   dimension=3072,
#   metric="cosine",
#   spec=PodSpec(
#     environment="gcp-starter",
#     pod_type="starter",
#     pods=1
#   ))

# index = pc.Index("alice")
# print(pc.list_indexes())

In [4]:
# import pandas as pd
# import numpy as np

# datafile_path = "data/Alice_with_embeddings.csv"
# df = pd.read_csv(datafile_path)
# #df["embedding"] = df.embedding.apply(eval).apply(np.array)
# df["embedding"] = df["embedding"].apply(eval)  # Convert string to Python list
# df["embedding"] = df["embedding"].apply(lambda arr: np.array(arr).tolist())  # Convert NumPy array to Python list

# df = df.rename(columns={"Unnamed: 0": "id", "embedding": "values", "paragraph": "paragraph"})

# dfvalues = df.copy()
# dfvalues.drop("paragraph", axis=1, inplace=True)
# dfvalues['id'] = dfvalues['id'].astype(str)
# dfvalues.head()

In [5]:
# index.upsert_from_dataframe(dfvalues[0:200])
# index.upsert_from_dataframe(dfvalues[201:400])
# index.upsert_from_dataframe(dfvalues[401:600])
# index.upsert_from_dataframe(dfvalues[601:800])
# index.upsert_from_dataframe(dfvalues[801:1000])
# index.upsert_from_dataframe(dfvalues[1001:1200])


In [6]:
from time import sleep

ready = False
while not ready:
    try:
        if pc.describe_index('alice')["status"]['ready']:
            ready = True
            print('Index is ready!')
        else:
            print('Index is not ready yet...')
            sleep(5)
    except pc.core.client.exceptions.NotFoundException:
        # NotFoundException means the index is created yet.
        pass
        

Index is ready!


In [7]:
index = pc.Index("alice")
index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 812}},
 'total_vector_count': 812}

In [18]:
from time import time
import pandas as pd
import numpy as np
datafile_path = "data/Alice_with_embeddings.csv"
df = pd.read_csv(datafile_path)
df["embedding"] = df["embedding"].apply(eval)  # Convert string to Python list
df["embedding"] = df["embedding"].apply(lambda arr: np.array(arr).tolist())  # Convert NumPy 
df = df.rename(columns={"Unnamed: 0": "id2", "embedding": "values", "paragraph": "paragraph"})

def searchdatabase(searchterm='Is Alice Happy?', results=3):
    import pandas as pd
    from openai import AzureOpenAI
    import configparser
    import time

    # Load API key
    config = configparser.RawConfigParser()
    config.read('keys/keys.ini')
    api_key = config['keys']['AzurekeyEUS']

    # Initialise Azure OpenAI client
    client = AzureOpenAI(
        azure_endpoint="https://eastusOpenAIral.openai.azure.com/",
        api_key=api_key,
        api_version="2025-03-01-preview"
    )

    # Get embedding for the search term
    response = client.embeddings.create(
        model="textembeddings3large",  # Ensure this matches your deployment name
        input=searchterm
    )
    query_embedding = response.data[0].embedding

    # Query the vector index (ensure index is defined globally or passed in)
    #print how long this call takes
    start_time = time.time()
    xc = index.query(
        vector=query_embedding,
        top_k=results,
        include_values=False
    )
    end_time = time.time()
    print(f"Query took {end_time - start_time} seconds")

    # Extract IDs
    ids = [x["id"] for x in xc["matches"]]
    dfres = pd.DataFrame(ids, columns=['id'])
    dfres['id'] = dfres['id'].astype(int)

    # Ensure df exists and has 'id' and 'paragraph' columns
    df['id'] = df['id'].astype(int)
    dffinalSearch = dfres.merge(df, on='id', how='left')

    return dffinalSearch['paragraph'].tolist()


In [21]:
searchdatabase('What can you eat in wonderland?', 5)

Query took 0.16974902153015137 seconds


["'What did they live on?' said Alice, who always took a great interest in\nquestions of eating and drinking.",
 'The great question certainly was, what? Alice looked all round her at\nthe flowers and the blades of grass, but she did not see anything that\nlooked like the right thing to eat or drink under the circumstances.\nThere was a large mushroom growing near her, about the same height as\nherself; and when she had looked under it, and on both sides of it, and\nbehind it, it occurred to her that she might as well look and see what\nwas on the top of it.',
 "The King and Queen of Hearts were seated on their throne when they\narrived, with a great crowd assembled about them--all sorts of little\nbirds and beasts, as well as the whole pack of cards: the Knave was\nstanding before them, in chains, with a soldier on each side to guard\nhim; and near the King was the White Rabbit, with a trumpet in one hand,\nand a scroll of parchment in the other. In the very middle of the court\nwas a